In [2]:
import pandas as pd
import re
from pypdf import PdfReader
from collections import defaultdict
import pdfplumber

In [49]:
PDF_PATH = r"C:\\Uni\\GP\\Data\\Hieroglyphs Translation\\Translation\\Dickson Dictionary.pdf"
CSV_PATH = r"C:\\Uni\\GP\\Data\\Hieroglyphs Translation\\Translation\\ancient egypt dictionary.csv" 

In [4]:
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[5]
    text = page.extract_text()
text 

'Dictionary of Middle Egyptian A - Men 6\n[qAyt] high ground, arable land {A28} [sAwty] guardian {A47 G1 G43\nD40 X1 Z4}\n[qAi] (v. infinitive) be raised on high, uplifted\n{A28 X1} [sAi] sift (flour etc) {A47 G1 Z7 A24}\n[qAi] (adj. and v.) tall, high, exalted, be raised [sAw] (v.) guard, ward off, restrain,\non high, uplifted {A28 Y1v} heed {A47 G1 Z7 A24}\n[qAt] height {A28 Y1v} [sAw] guardian, warden {A47 G43}\n[iAw] adoration {A30} [sAw] (v.) guard, ward off, restrain, heed\n{A47 G43 A24}\n[xwsi] pound, beat up, beat flat, build up, build,\nconstruct, stir {A34} [mniw] herdsman {A47 G43 D40}\n[qd] builder {A35} [sAi iit .f] one whose coming is\nawaited {A47 G43 D40 D21 M18 M17 X1 I9}\n[qd] build, fashion (men) {A35 A24}\n[sAi] (v.) await {A47 M17 M17}\n[afty] brewer {A36 X1 Z4 A24}\n[iry sSm] functionary {A47 S29 T32}\n[afty] brewer {A37 X1 Z4}\n[irt] duty (of someone) {A47 X1 Z2}\n[qis] (locality) Cusae (El - Kusiyah) {A38}\n[irt] duty (of someone) {A47 Y1 Z2}\n[qis] (locality) C

In [5]:
def clean_text(text):
    text = text or ""
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_definition(text):
    # remove part of speech: (v.), (n.), (suffix prn.), etc.
    text = re.sub(r"\([^)]*\)", " ", text)

    # remove extra spaces around hyphens/punctuation
    text = re.sub(r"\s*-\s*", "-", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip(" ,.;:")


def split_definitions(definition):
    parts = [p.strip(" ,.;:") for p in definition.split(",")]
    return [p for p in parts if p]


def normalize_codes(codes):
    codes = re.sub(r"\s+", " ", codes.strip())

    # Normalize common OCR/text extraction issue
    codes = codes.replace("Y1V", "Y1v")

    return codes


def parse_column_text(text):
    """
    Parses entries like:
    [qAi] (v. infinitive) be raised on high, uplifted {A28 X1}
    """

    text = clean_text(text)

    # Find complete dictionary entries only:
    # starts with [transliteration], ends at {Gardiner sequence}
    pattern = re.compile(
        r"\[[^\[\]{}]+\]\s+(.*?)\s*\{([^{}]+)\}",
        re.DOTALL
    )

    entries = []

    for match in pattern.finditer(text):
        raw_definition = match.group(1)
        raw_codes = match.group(2)

        definition = clean_definition(raw_definition)
        codes = normalize_codes(raw_codes)

        # Basic validation: Gardiner codes should look like A1, D21, Aa1, Y1v, etc.
        code_tokens = codes.split()
        valid_code_seq = all(
            re.fullmatch(r"(?:Aa|[A-Z])[0-9]+[A-Za-z]?", token)
            for token in code_tokens
        )

        if not valid_code_seq:
            continue

        # Avoid obvious corrupted definitions containing another entry
        if "[" in definition or "]" in definition or "{" in definition or "}" in definition:
            continue

        entries.append((codes, definition))

    return entries


def extract_page_columns(page):
    """
    Splits a page into left and right columns.
    This avoids pdfplumber mixing entries from both columns.
    """

    width = page.width
    height = page.height

    # remove header/footer margins
    top = 40
    bottom = height - 30

    mid = width / 2

    left_bbox = (0, top, mid, bottom)
    right_bbox = (mid, top, width, bottom)

    left_text = page.crop(left_bbox).extract_text(x_tolerance=1, y_tolerance=3) or ""
    right_text = page.crop(right_bbox).extract_text(x_tolerance=1, y_tolerance=3) or ""

    return left_text, right_text


def build_gardiner_dictionary(pdf_path, start_page=4):
    """
    start_page=4 means page index 4, i.e. printed page 5,
    where the dictionary entries begin.
    """

    gardiner_dict = defaultdict(list)

    with pdfplumber.open(pdf_path) as pdf:
        pending_left = ""
        pending_right = ""

        for page_index in range(start_page, len(pdf.pages)):
            page = pdf.pages[page_index]
            left_text, right_text = extract_page_columns(page)

            # Join with pending text from previous page/column
            for column_text in [pending_left + " " + left_text,
                                pending_right + " " + right_text]:

                entries = parse_column_text(column_text)

                for codes, definition in entries:
                    for item in split_definitions(definition):
                        if item not in gardiner_dict[codes]:
                            gardiner_dict[codes].append(item)

            # Keep last incomplete entry from each column, if any
            pending_left = get_unfinished_tail(left_text)
            pending_right = get_unfinished_tail(right_text)

    return dict(gardiner_dict)


def get_unfinished_tail(text):
    """
    Keeps the last entry fragment if it starts with '[' but has no closing {codes}.
    This helps when an entry is split across pages.
    """

    text = clean_text(text)

    last_entry_start = text.rfind("[")

    if last_entry_start == -1:
        return ""

    tail = text[last_entry_start:]

    # If it already contains a complete code block, don't keep it
    if re.search(r"\{[^{}]+\}", tail):
        return ""

    return tail

In [50]:
def normalize_code_sequence(codes):
    codes = str(codes).strip()
    codes = codes.replace("-", " ")
    codes = re.sub(r"\s+", " ", codes)
    return codes


def valid_gardiner_sequence(codes):
    tokens = codes.split()
    return bool(tokens) and all(
        re.fullmatch(r"(?:Aa|[A-Z])[0-9]+[A-Za-z]?", tok)
        for tok in tokens
    )


def clean_translation(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" ,.;:")


def split_translation(text):
    return [
        part.strip(" ,.;:")
        for part in text.split(",")
        if part.strip(" ,.;:")
    ]

def merge_csv_dictionary(gardiner_dict, csv_path):
    df_new = pd.read_csv(csv_path)

    # normalize column names
    df_new.columns = [c.strip().lower() for c in df_new.columns]

    if "gardiner" not in df_new.columns or "translation" not in df_new.columns:
        raise ValueError("CSV must contain columns: gardiner, translation")

    added_codes = 0
    added_translations = 0
    skipped_rows = 0

    for _, row in df_new.iterrows():
        codes = normalize_code_sequence(row["gardiner"])
        translation = clean_translation(row["translation"])

        if not valid_gardiner_sequence(codes):
            skipped_rows += 1
            continue

        if not translation or translation.lower() == "nan":
            skipped_rows += 1
            continue

        if codes not in gardiner_dict:
            gardiner_dict[codes] = []
            added_codes += 1

        for meaning in split_translation(translation):
            if meaning not in gardiner_dict[codes]:
                gardiner_dict[codes].append(meaning)
                added_translations += 1

    print("New code sequences added:", added_codes)
    print("New translations added:", added_translations)
    print("Skipped rows:", skipped_rows)

    return gardiner_dict

In [45]:
gardiner_dict = build_gardiner_dictionary(PDF_PATH)

print("Total code sequences:", len(gardiner_dict))

Total code sequences: 10629


In [51]:
gardiner_dict = merge_csv_dictionary(gardiner_dict, CSV_PATH)

New code sequences added: 716
New translations added: 5464
Skipped rows: 18


In [53]:
for k in list(gardiner_dict.keys())[:10]:
    print(k, "=>", gardiner_dict[k])

A1 => ['I', 'me', 'my']
A1 A1 A1 => ['man', 'men', 'mankind', 'Egyptians']
A2 => ['drink']
A6 => ['pure', 'purify oneself', 'bathe', 'cleanse', 'purification', 'purity']
A6A => ['pure', 'purify oneself', 'bathe', 'cleanse', 'purification', 'purity']
A6 D58 G43 X1 => ['priestly service']
A7 => ['weariness', 'languor', 'slackness', 'remissness', 'sit down']
A8 A1 B1 Z2 => ['associates', 'family']
A9 A24 => ['be heavy laden', 'carry', 'support', 'be heavy laden with trouble']
A9 Q3 Z7 Y1 Z2 => ['cargo']


In [54]:
def tokenize_codes(code_sequence):
    if isinstance(code_sequence, str):
        return code_sequence.strip().split()
    return list(code_sequence)


def extract_longest_matches(code_sequence, gardiner_dict, max_len=None):
    """
    Returns non-overlapping matches, preferring the longest valid Gardiner-code sequences.
    """

    codes = tokenize_codes(code_sequence)
    n = len(codes)

    if max_len is None:
        max_len = max(len(k.split()) for k in gardiner_dict.keys())

    matches = []
    i = 0

    while i < n:
        best_match = None

        for j in range(min(n, i + max_len), i, -1):
            seq = " ".join(codes[i:j])

            if seq in gardiner_dict:
                best_match = {
                    "start": i,
                    "end": j,
                    "codes": seq,
                    "candidates": gardiner_dict[seq],
                    "matched_tokens": codes[i:j]
                }
                break

        if best_match:
            matches.append(best_match)
            i = best_match["end"]
        else:
            matches.append({
                "start": i,
                "end": i + 1,
                "codes": codes[i],
                "candidates": [],
                "matched_tokens": [codes[i]]
            })
            i += 1

    return matches

def extract_overlapping_matches(code_sequence, gardiner_dict, max_len=None):
    codes = code_sequence.strip().split()
    n = len(codes)

    if max_len is None:
        max_len = max(len(k.split()) for k in gardiner_dict)

    all_matches = []

    for i in range(n):
        local_matches = []

        for j in range(i + 1, min(n, i + max_len) + 1):
            seq = " ".join(codes[i:j])

            if seq in gardiner_dict:
                local_matches.append({
                    "start": i,
                    "end": j,
                    "codes": seq,
                    "length": j - i,
                    "candidates": gardiner_dict[seq]
                })

        if local_matches:
            local_matches = sorted(
                local_matches,
                key=lambda x: (-x["length"], x["start"])
            )
            all_matches.extend(local_matches)

    return all_matches

In [55]:
def filter_overlapping_matches(matches, min_len=2):
    return [
        m for m in matches
        if m["length"] >= min_len
    ]

In [56]:
#code_sequence = "M17 G43 D4 N35 M17 M40 O34 O1 Z1 G17 V28 W14 X1 O34 M23 X1 N35 S29 M17 N29 D21 N35 G43 M17 U36 Z1 I9 G17 N17 N23 S29 V13 N35 G41 V31 A1 D21 S38 N29 S38 N29 S38 N29 W24 Z1 N24C X1 Z2"
code_sequence = "W25 N35 N35 N35 O4 G17 T3 G17 N14 Z1 Aa1 D58 D58 A32G M17 G17 W11 N35 F32 X1 N14 Z2"
#code_sequence = "E34 N35 V31 D21 I9 G17 D54 G43 I9 O29 D36 I9"
#code_sequence = "F18 D46 X1 R4"
#code_sequence = "N35 V24 G43 V28 G43 X1 V1 G17 O34 A1 A1"
#code_sequence = "N35 N26 G43 S29 O34 A1 N35 D2 D1 I9"

In [57]:
greedy_matches = extract_longest_matches(code_sequence, gardiner_dict)

for m in greedy_matches:
    print(m["codes"], "=>", m["candidates"])

W25 N35 => ['bring', 'fetch', 'carry off', 'remove', 'overcome', 'reach', 'attain', 'buy', 'appoint', 'use']
N35 => ['to', 'for', 'in', 'because', 'belongs to', 'we', 'us', 'our', 'of', 'belonging to', 'to persons', 'in sun', 'dew', 'time']
N35 => ['to', 'for', 'in', 'because', 'belongs to', 'we', 'us', 'our', 'of', 'belonging to', 'to persons', 'in sun', 'dew', 'time']
O4 => []
G17 => ['do not', 'in the position of', 'with', 'by means of', 'from', 'out of', 'as', 'namely', 'when', 'though', 'together with']
T3 => ['white', 'bright']
G17 => ['do not', 'in the position of', 'with', 'by means of', 'from', 'out of', 'as', 'namely', 'when', 'though', 'together with']
N14 Z1 => ['star']
Aa1 => []
D58 => ['place']
D58 => ['place']
A32G => []
M17 G17 => ['in', 'with', 'by means of', 'from', 'out of', 'as', 'namely', 'when', 'though', 'together with', 'there', 'therein', 'therewith', 'therefrom']
W11 N35 F32 X1 N14 => ['star']
Z2 => []


In [58]:
overlapping_matches = extract_overlapping_matches(code_sequence, gardiner_dict)

for m in overlapping_matches:
    print(m["codes"], "=>", m["candidates"])

W25 N35 => ['bring', 'fetch', 'carry off', 'remove', 'overcome', 'reach', 'attain', 'buy', 'appoint', 'use']
W25 => ['bring', 'fetch', 'carry off', 'bring away', 'bring about', 'remove', 'overcome', 'reach', 'attain', 'buy', 'appoint', 'use', 'remove something', 'overcome trouble', 'attain a place']
N35 => ['to', 'for', 'in', 'because', 'belongs to', 'we', 'us', 'our', 'of', 'belonging to', 'to persons', 'in sun', 'dew', 'time']
N35 => ['to', 'for', 'in', 'because', 'belongs to', 'we', 'us', 'our', 'of', 'belonging to', 'to persons', 'in sun', 'dew', 'time']
N35 => ['to', 'for', 'in', 'because', 'belongs to', 'we', 'us', 'our', 'of', 'belonging to', 'to persons', 'in sun', 'dew', 'time']
G17 => ['do not', 'in the position of', 'with', 'by means of', 'from', 'out of', 'as', 'namely', 'when', 'though', 'together with']
T3 => ['white', 'bright']
G17 => ['do not', 'in the position of', 'with', 'by means of', 'from', 'out of', 'as', 'namely', 'when', 'though', 'together with']
N14 Z1 => ['s

In [59]:
filtered_overlaps = filter_overlapping_matches(overlapping_matches, min_len=2)
for m in filtered_overlaps:
    print(m["codes"], "=>", m["candidates"])

W25 N35 => ['bring', 'fetch', 'carry off', 'remove', 'overcome', 'reach', 'attain', 'buy', 'appoint', 'use']
N14 Z1 => ['star']
M17 G17 => ['in', 'with', 'by means of', 'from', 'out of', 'as', 'namely', 'when', 'though', 'together with', 'there', 'therein', 'therewith', 'therefrom']
W11 N35 F32 X1 N14 => ['star']


In [61]:
def build_llm_prompt(code_sequence, greedy_matches, filtered_matches):
    lines = []

    lines.append("You are translating Middle Egyptian Gardiner-code sequences into fluent English.")
    lines.append("")
    lines.append("Rules:")
    lines.append("1. Produce ONE coherent English sentence.")
    lines.append("2. Use the dictionary candidates as your main evidence.")
    lines.append("3. Prefer the filtered overlapping matches as the main reading.")
    lines.append("4. Use greedy segmentations only when they improve the meaning.")
    lines.append("5. Ignore unmatched signs if they are likely determinatives, phonetic complements, plural markers, or grammatical signs.")
    lines.append("6. Do not list dictionary candidates in the final translation.")
    lines.append("7. Do not output word-by-word translation.")
    lines.append("8. If a necessary linking word is needed for fluent English, you may add it, but keep the meaning grounded.")
    lines.append("")
    lines.append("Input Gardiner sequence:")
    lines.append(code_sequence)
    lines.append("")
    lines.append("Filtered overlapping alternatives:")

    for m in filtered_matches:
        candidates = ", ".join(m["candidates"]) if m["candidates"] else "[NO MATCH]"
        lines.append(f"- {m['codes']} => {candidates}")
    lines.append("")
    lines.append("Primary greedy segmentation:")

    for m in greedy_matches:
        candidates = ", ".join(m["candidates"]) if m["candidates"] else "[NO MATCH]"
        lines.append(f"- {m['codes']} => {candidates}")

    lines.append("")
    lines.append("Output format:")
    lines.append("Translation: <one fluent English sentence>")
    lines.append("Used evidence: <short list of the most important code-to-meaning choices>")
    lines.append("Uncertain signs: <codes that were unmatched or uncertain>")

    return "\n".join(lines)

In [62]:
prompt = build_llm_prompt(
    code_sequence,
    greedy_matches,
    filtered_overlaps
)

In [63]:
prompt

'You are translating Middle Egyptian Gardiner-code sequences into fluent English.\n\nRules:\n1. Produce ONE coherent English sentence.\n2. Use the dictionary candidates as your main evidence.\n3. Prefer the filtered overlapping matches as the main reading.\n4. Use greedy segmentations only when they improve the meaning.\n5. Ignore unmatched signs if they are likely determinatives, phonetic complements, plural markers, or grammatical signs.\n6. Do not list dictionary candidates in the final translation.\n7. Do not output word-by-word translation.\n8. If a necessary linking word is needed for fluent English, you may add it, but keep the meaning grounded.\n\nInput Gardiner sequence:\nW25 N35 N35 N35 O4 G17 T3 G17 N14 Z1 Aa1 D58 D58 A32G M17 G17 W11 N35 F32 X1 N14 Z2\n\nFiltered overlapping alternatives:\n- W25 N35 => bring, fetch, carry off, remove, overcome, reach, attain, buy, appoint, use\n- N14 Z1 => star\n- M17 G17 => in, with, by means of, from, out of, as, namely, when, though, tog

In [64]:
import os
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)

In [65]:
##gpt-oss-120b
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": (
                "You must ONLY use the provided candidate meanings. "
                "Do not invent translations. "
                "If uncertain, say uncertain."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

Translation: Bring it to the white star with the place, in the star.  
Used evidence: W25 N35 → bring; N35 → to; T3 → white; N14 Z1 → star; G17 → with; D58 → place; M17 G17 → in; W11 N35 F32 X1 N14 → star.  
Uncertain signs: O4, Aa1, A32G, Z2.


In [66]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    messages=[
        {
            "role": "system",
            "content": (
                "You must ONLY use the provided candidate meanings. "
                "Do not invent translations. "
                "If uncertain, say uncertain."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

Translation: They bring the star to the place with them.
Used evidence: W25 N35, N14 Z1, M17 G17, W11 N35 F32 X1 N14, D58 D58
Uncertain signs: O4, Aa1, A32G, Z2, G17 (when not part of M17 G17), T3, N35 (when not part of W25 N35 or W11 N35 F32 X1 N14)


In [81]:
def extract_translation(output):
    # Match: Translation:, **Translation:**, etc.
    match = re.search(
        r"\*{0,2}\s*translation\s*:\s*\*{0,2}\s*(.+)",
        output,
        re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    # fallback: return first non-empty line
    for line in output.split("\n"):
        line = line.strip()
        if line:
            return line

    return ""

def generate_rag_prediction(code_sequence, gardiner_dict, llm_client):
    greedy = extract_longest_matches(code_sequence, gardiner_dict)
    overlapping = extract_overlapping_matches(code_sequence, gardiner_dict)
    filtered = filter_overlapping_matches(overlapping, min_len=2)

    prompt = build_llm_prompt(code_sequence, greedy, filtered)

    response = llm_client.chat.completions.create(
        #model="openai/gpt-oss-120b",
        model="llama-3.3-70b-versatile",
        temperature=0.1,
        messages=[
            {"role": "system", "content": "You are a precise translator."},
            {"role": "user", "content": prompt}
        ]
    )

    output = response.choices[0].message.content

    # Extract only the translation line
    return extract_translation(output)

In [68]:
import evaluate

bleu = evaluate.load("bleu")

def compute_bleu(predictions, references):
    """
    predictions: list of model outputs
    references: list of ground-truth translations
    """

    predictions = [p.strip() for p in predictions]
    references = [[r.strip()] for r in references]

    result = bleu.compute(
        predictions=predictions,
        references=references
    )

    return result

c:\Users\georg\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).
W0429 00:42:46.788000 6184 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [69]:
def evaluate_rag_bleu(df, prediction_col="Predicted_Translation", reference_col="Translation"):
    predictions = df[prediction_col].fillna("").astype(str).tolist()
    references = df[reference_col].fillna("").astype(str).tolist()

    return compute_bleu(predictions, references)

In [82]:
df = pd.read_excel("C:\\Uni\\GP\\Data\\Hieroglyphs Translation\\Translation\\Code and Translation.xlsx")

In [83]:
df = df[df["Translation"].str.len() > 0]
df = df[df["Code"].str.len() > 0]

In [84]:
def clean_text_column(col):
    return col.fillna("").astype(str).str.strip()
df['Code'] = clean_text_column(df['Code'])
df['Translation'] = clean_text_column(df['Translation'])

In [85]:
def clean_gardiner(text):
    # 1. Remove anything inside square brackets (including brackets)
    text = re.sub(r"\[.*?\]", "", text)

    # 2. Find Gardiner codes:
    # Pattern: letters (1–3) + numbers (e.g., D21, Aa17, Z1, X1, etc.)
    codes = re.findall(r"\b[A-Za-z]{1,3}\d+\b", text)

    return " ".join(codes)
df['Code'] = df['Code'].apply(clean_gardiner)

In [86]:
# Work on a copy
df_clean = df.copy()

# Ensure strings
df_clean["Code"] = df_clean["Code"].fillna("").astype(str)
df_clean["Translation"] = df_clean["Translation"].fillna("").astype(str)

def clean_translation(text):
    text = str(text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Remove bracketed damaged/unknown parts inside sentences
    text = re.sub(r"\[\s*\.\.\.\s*\]", " ", text)
    text = re.sub(r"\[\s*---+\s*\]", " ", text)

    # Remove standalone damage markers
    text = re.sub(r"--destroyed--", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bunknown\b", " ", text, flags=re.IGNORECASE)

    # Remove repeated dashes / ellipses anywhere
    text = re.sub(r"\.{2,}", " ", text)
    text = re.sub(r"-{2,}", " ", text)

    # Remove empty square brackets left behind
    text = re.sub(r"\[\s*\]", " ", text)

    # Normalize quotation marks
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("’", "'").replace("‘", "'")

    # Remove extra spaces before punctuation
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)

    # Normalize spaces again
    text = re.sub(r"\s+", " ", text).strip()

    return text


df_clean["Translation_clean"] = df_clean["Translation"].apply(clean_translation)

# Remove rows only if translation became empty or useless
bad_exact = {
    "",
    ".",
    ",",
    ";",
    ":",
    "-",
    "...",
    "[...]",
    "[---]",
}

df_clean = df_clean[
    ~df_clean["Translation_clean"].str.strip().isin(bad_exact)
]

# Remove rows with no Gardiner code
df_clean = df_clean[df_clean["Code"].str.strip().ne("")]

# Optional: remove very short targets that are likely fragments
# Use this carefully; you may comment it out if short translations are valid.
df_clean = df_clean[
    df_clean["Translation_clean"].str.split().str.len() >= 2
]

# Replace original column
df_clean["Translation"] = df_clean["Translation_clean"]
df_clean = df_clean.drop(columns=["Translation_clean"])

# Remove exact duplicate input-output pairs
df_clean = df_clean.drop_duplicates(subset=["Code", "Translation"])

# Reset index
df_clean = df_clean.reset_index(drop=True)

print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Removed rows:", len(df) - len(df_clean))

Original rows: 37595
Cleaned rows: 33220
Removed rows: 4375


In [87]:
df_clean.head()

,Code,Translation
0,D21 Q3 D36 F4 D36 L2 X1 S19 S29 U23 T21 X1 G17...,"Hereditary noble and prince, royal seal-bearer..."
1,M17 S34 Aa1 G43 A1 N17 A1 S29 V4 X1 S29 N35 D2...,"O living ones, who are upon the earth, who sha..."
2,G35 F34 F34 F34 D2 Z1 Aa17 U6 D21 M17 M17 X1 N...,"A trusted one upon the landing place,great ove..."
3,M17 G43 D4 N35 M17 M40 O34 O1 Z1 G17 V28 W14 X...,"I built a tomb through the favour of the king,..."
4,M17 G43 N35 O4 Q3 Y2 Z2 W24 Z1 M17 M2 O34 X8 F...,"I restored the laws of the ancient times, it w..."


In [88]:
import time

for idx, row in df_clean.iloc[0:86].iterrows():
    code_seq = row["Code"]
    reference = row["Translation"]

    predicted = generate_rag_prediction(code_seq, gardiner_dict, client)

    df_clean.at[idx, "Predicted_Translation"] = predicted
    if idx % 20 == 0:
        print(f"Processed {idx} entries...")
        time.sleep(62)

Processed 0 entries...
Processed 20 entries...
Processed 40 entries...
Processed 60 entries...
Processed 80 entries...


In [89]:
df_clean[df_clean["Predicted_Translation"].notna()].count()

Code                     86
Translation              86
Predicted_Translation    86
dtype: int64

In [78]:
# gpt-oss-120b had 0.023 BLEU
bleu_result = evaluate_rag_bleu(
    df_clean.iloc[0:86],
    prediction_col="Predicted_Translation",
    reference_col="Translation"
)

print(bleu_result)

{'bleu': 0.019904545105105528, 'precisions': [0.3128834355828221, 0.04561911658218682, 0.0069498069498069494, 0.0024813895781637717], 'brevity_penalty': 0.8936201898684453, 'length_ratio': 0.8988970588235294, 'translation_length': 1467, 'reference_length': 1632}


In [90]:
#llama-3.3-70b-versatile had 0.025 BLEU
bleu_result = evaluate_rag_bleu(
    df_clean.iloc[0:86],
    prediction_col="Predicted_Translation",
    reference_col="Translation"
)

print(bleu_result)

{'bleu': 0.02478981305257713, 'precisions': [0.2382159148504815, 0.04080551139374669, 0.00832870627429206, 0.004664723032069971], 'brevity_penalty': 1.0, 'length_ratio': 1.2089460784313726, 'translation_length': 1973, 'reference_length': 1632}


In [79]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

def semantic_similarity(preds, refs):
    pred_emb = model.encode(preds)
    ref_emb = model.encode(refs)
    
    sims = cosine_similarity(pred_emb, ref_emb)
    return sims.diagonal().mean()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3029.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [80]:
#gpt-oss-120b had 0.286 semantic similarity
semantic_result = semantic_similarity(
    df_clean['Predicted_Translation'].iloc[0:86].tolist(),
    df_clean['Translation'].iloc[0:86].tolist()
)
semantic_result

np.float32(0.29092792)

In [91]:
semantic_result = semantic_similarity(
    df_clean['Predicted_Translation'].iloc[0:86].tolist(),
    df_clean['Translation'].iloc[0:86].tolist()
)
semantic_result

np.float32(0.276844)